# 알약 검출 베이스라인 — Google Colab

**사전 준비:** 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택

## 세션이 끊겼으면 셀 1~6번만 / 전처리·변환은 파일이 있으면 자동 스킵

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 & 실행 설정 — **여기만 수정하세요**

| 변수 | 설명 |
|---|---|
| `PROJECT_DIR` | 구글 드라이브 프로젝트 루트 |
| `RUN_NAME` | 학습 이름 → `outputs/yolo/<RUN_NAME>/weights/best.pt` |
| `SUBMISSION_NAME` | 캐글 제출 CSV 파일명 |

In [ ]:
import os

# ─── 여기만 수정 ───────────────────────────────────────
PROJECT_DIR     = '/content/drive/MyDrive/베이스라인_코랩버전v1.2'  # ← Drive 경로
RUN_NAME        = 'train_m'        # outputs/yolo/<RUN_NAME>/
SUBMISSION_NAME = 'submission_yolo.csv'  # 캐글 제출 CSV 이름
# ────────────────────────────────────────────────────

BASELINE_DIR = os.path.join(PROJECT_DIR, 'baseline')
DATA_DIR     = os.path.join(PROJECT_DIR, 'sprint_ai_project1_data')

BEST_WEIGHTS    = f'outputs/yolo/{RUN_NAME}/weights/best.pt'
SUBMISSION_PATH = f'outputs/predictions/{SUBMISSION_NAME}'

for p in [PROJECT_DIR, BASELINE_DIR, DATA_DIR]:
    print(f'exists={os.path.exists(p)}  {p}')
print(f'\nRUN_NAME        = {RUN_NAME}')
print(f'BEST_WEIGHTS    = {BEST_WEIGHTS}')
print(f'SUBMISSION_NAME = {SUBMISSION_NAME}')

## 3. 작업 디렉토리 설정

In [ ]:
import os

os.chdir(BASELINE_DIR)
print('현재 디렉토리:', os.getcwd())
print('내용:', os.listdir('.'))

## 4. config 경로 업데이트

In [ ]:
import yaml

cfg_path = 'configs/default.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['data_root']     = DATA_DIR
cfg['data']['processed_dir'] = './data/processed'
cfg['data']['num_workers']   = 2

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('data_root:', cfg['data']['data_root'])

## 5. 패키지 설치

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ultralytics ensemble-boxes

## 6. GPU 확인

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 6-b. 데이터 정합성 필터 — bbox 수가 안 맞는 이미지 제거

이미지 파일명 prefix(예: `K-001900-003544-016548_..._.png`)에서 첫 `_` 앞의 `-` 개수가
실제 알약 개수입니다. 해당 이미지의 JSON 파일 수와 불일치하면 **Drive에서 삭제**합니다.

> 최초 1회만 의미 있음. 삭제된 파일은 Drive에서 영구 제거됩니다.

In [ ]:
import os
from pathlib import Path

TRAIN_IMG_DIR = Path(DATA_DIR) / 'train_images'
TRAIN_ANN_DIR = Path(DATA_DIR) / 'train_annotations'

images = sorted(TRAIN_IMG_DIR.glob('*.png'))
print(f'[scan] {len(images)} images')

def expected_count(fname: str) -> int:
    # 'K-001900-003544-016548_0_2_...png' → 첫 _ 앞 대시 개수 = 알약 수
    return fname.split('_', 1)[0].count('-')

def find_json_files(img_path: Path):
    prefix = img_path.name.split('_', 1)[0]
    ann_dir = TRAIN_ANN_DIR / f'{prefix}_json'
    stem = img_path.stem
    if not ann_dir.exists():
        return []
    return list(ann_dir.glob(f'*/{stem}.json'))

dropped_imgs, dropped_jsons = [], 0
kept = 0
for img in images:
    exp = expected_count(img.name)
    jsons = find_json_files(img)
    got = len(jsons)
    if exp == got:
        kept += 1
        continue
    img.unlink()
    for j in jsons:
        j.unlink()
        dropped_jsons += 1
    dropped_imgs.append((img.name, exp, got))

print(f'kept:    {kept}')
print(f'dropped: {len(dropped_imgs)} images, {dropped_jsons} json files')
for fn, exp, got in dropped_imgs[:20]:
    print(f'  expected={exp} got={got}  {fn}')
if len(dropped_imgs) > 20:
    print(f'  ... and {len(dropped_imgs)-20} more')

## 7. 어노테이션 전처리 — 최초 1회만 실행, 이후 자동 스킵

> `data/processed/annotations.json` 이 Drive에 이미 있으면 스킵합니다.

In [ ]:
import os

if os.path.exists('data/processed/annotations.json'):
    print('[skip] 어노테이션 전처리 이미 완료 → 스킵')
else:
    print('[run] 어노테이션 전처리 중...')
    !python scripts/preprocess.py

## 8. YOLO 데이터 변환 — 최초 1회만 실행, 이후 자동 스킵

> `data/yolo/dataset.yaml` 이 Drive에 이미 있으면 스킵합니다.

In [ ]:
import os

if os.path.exists('data/yolo/dataset.yaml'):
    print('[skip] YOLO 변환 이미 완료 → 스킵')
else:
    print('[run] YOLO 형식 변환 중...')
    !python scripts/convert_to_yolo.py

---
## [YOLO11] 학습

모델 크기: `yolo11n`(빠름) / `yolo11s` / `yolo11m`(추천) / `yolo11l` / `yolo11x`(최고성능)

증강 옵션: `--mosaic` `--mixup` `--copy_paste` `--degrees` `--flipud` (0.0~1.0)

In [ ]:
!python train_yolo.py \
    --model yolo11m.pt \
    --epochs 60 \
    --batch 8 \
    --imgsz 1280 \
    --output outputs/yolo \
    --name {RUN_NAME}

In [ ]:
# (선택) 두 번째 모델 학습 — 앙상블용
!python train_yolo.py \
    --model yolo11l.pt \
    --epochs 60 \
    --batch 4 \
    --imgsz 1280 \
    --output outputs/yolo \
    --name train_l

## [YOLO11] 단일 모델 추론 → 캐글 제출 CSV

In [ ]:
import os

!python inference_yolo.py \
    --checkpoint {BEST_WEIGHTS} \
    --conf 0.2

default_csv = 'outputs/predictions/submission_yolo.csv'
if SUBMISSION_PATH != default_csv and os.path.exists(default_csv):
    os.replace(default_csv, SUBMISSION_PATH)
    print(f'[rename] → {SUBMISSION_PATH}')

## [YOLO11] 앙상블 추론 (두 모델 학습 후 사용)

`train_m`, `train_l` 두 모델을 각각 학습한 뒤 실행하세요.

In [ ]:
!python inference_yolo_ensemble.py \
    --checkpoints outputs/yolo/train_m/weights/best.pt \
                  outputs/yolo/train_l/weights/best.pt \
    --conf 0.2

---
## 틀린 이미지 시각화

In [ ]:
# 한글 폰트 설치 (최초 1회)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
print('폰트 설치 완료')

In [ ]:
!python scripts/visualize_errors.py --n 10 --output outputs/errors

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

error_files = sorted(glob.glob('outputs/errors/*.jpg'))
print(f'틀린 이미지 {len(error_files)}개')

for path in error_files[:10]:
    fig, ax = plt.subplots(1, figsize=(6, 8))
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=9)
    plt.tight_layout()
    plt.show()

---
## Drive 백업

학습 가중치(`outputs/yolo/`)와 제출 CSV(`outputs/predictions/`)만 백업합니다.

In [ ]:
import shutil, os, time

BACKUP_DIR = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(BACKUP_DIR, exist_ok=True)

for folder in ['predictions', 'yolo']:
    src = os.path.join('outputs', folder)
    if not os.path.exists(src):
        continue
    dst = os.path.join(BACKUP_DIR, folder)
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[ok] {folder} → Drive ({time.time()-t0:.1f}s)')

print('백업 완료:', BACKUP_DIR)